In [ ]:
# hide
import numpy as np
import pyquist as pq
from pyquist.helper import amplitude_to_db, db_to_amplitude

In [ ]:
# collapse
def multitap_delay(x: np.ndarray, taps: list[np.ndarray]) -> np.ndarray:
    """Read one delay line at several time-varying, fractional delays.
    Returns one row per tap, where row t is x[n - taps[t][n]]."""
    L = int(np.ceil(max(M.max() for M in taps))) + 2
    buffer = np.zeros(L)              # the delay line, as a circular buffer
    i = 0
    y = np.zeros((len(taps), len(x)))
    for n in range(len(x)):
        buffer[i] = x[n]              # write first, so that a delay of 0 reads x[n]
        for t, M in enumerate(taps):
            k = int(M[n])
            a = M[n] - k
            y[t, n] = (1 - a) * buffer[(i - k) % L] + a * buffer[(i - k - 1) % L]
        i = (i + 1) % L
    return y

def wet_ratio(db: float) -> float:
    """The wet ratio w that mixes the wet signal `db` decibels above the dry signal."""
    return 1 / (1 + db_to_amplitude(-db))

In [ ]:
def chorus(x: np.ndarray, delays: list[float], depth: float, rates: list[float],
           g: float = 0.7, f_s: int = 44100) -> np.ndarray:
    """The dry signal plus one delayed copy per voice, whose delay (in seconds)
    wobbles by up to `depth` seconds around its own base delay, at its own rate."""
    n = np.arange(len(x))
    taps = [f_s * (d + depth * np.sin(2 * np.pi * r * n / f_s)) for d, r in zip(delays, rates)]
    return x + g * multitap_delay(x, taps).sum(axis=0)


f_s = 44100
guitar = pq.Audio.from_file("../assets/guitar-arpeggio.wav").samples[:, 0]
wet = chorus(guitar, delays=[0.020, 0.025, 0.030], depth=0.003, rates=[0.35, 0.5, 0.65])
w = wet_ratio(0.0)
pq.play(pq.Audio(guitar, f_s), normalize=True)
pq.play(pq.Audio((1 - w) * guitar + w * wet, f_s), normalize=True)